# Concept-mass grounding — C · T4×2 · endpoint500
**Internet ON · T4×2 · Save & Run All. Mỗi notebook một nhánh, dùng cả hai GPU.**

C: native A + SCST reward gốc + rollout-KL + giữ xác suất khái niệm đúng.
G: như C, thêm phạt probability mass của nhóm cách gọi đối tượng chưa được annotation hỗ trợ.
Chưa có kết quả; đây là ứng viên objective ở mức khái niệm, chưa phải căn chỉnh đúng vùng/instance.

Add Input: `pair-source-fullframe-ce`, `prompt-cache`, `visual-cache`, `objectdetectionecache`,
`mscoco-2014` của vuthetam. Không cần RGB, UPT, VSR, checkpoint I/J hoặc dataset mới.
Bản v2 giữ output cũ và dùng thư mục output mới. Tự tìm stage cũ nếu còn trong Working.
Stage cũ chưa có row journal phải đọc lại nguồn để xác minh từng row, không tin mặc định4.750rows.
Đọc lại có thể vẫn tốn I/O; không có cam kết khôi phục tức thì. Dữ liệu/500updates/loss không đổi.
GT TRAIN10k và GT dev-only diagnostic nhỏ được pin trong Git; nhãn dev không dùng trong training.

Policy đã sửa trước run: GT/ref absence là nhãn yếu, detector chỉ veto mọi hint hợp lệ có sẵn.
Không suy vắng vật từ cache; nguồn extractor detector vẫn unknown. Không gọi negative là verified absence.
Nếu không có usable negative events, G chưa kiểm được objective, không báo đã có đối chứng thành công.

TRAIN population10k; stage đúng4000images trong lịch500updates, batch8 và fixeddev1000.
Cả hai học hai decoder layer cuối từ cùng A; không thêm model head, CE warmup hoặc sửa logits inference.
Giới hạn CPUprepare60phút, GPUphase60phút, CPUevaluation15phút/nhánh; đây là caps, không ETA.
User báo24giờ còn lại; không tự cộng quota hai tài khoản. Notebook chuẩn bị/CPUmetrics vẫn có thể tiêu GPU quota khi accelerator bật.


In [ ]:
from pathlib import Path
ARM='C'
CHECKPOINT=Path('/kaggle/input/datasets/ducanh2403/pair-source-fullframe-ce/upt_hidden_caption_A/checkpoint.bin')
COCO_JSON=Path('/kaggle/input/datasets/vuthetam/mscoco-2014/dataset_coco.json')
LEGACY_PROMPT=Path('/kaggle/input/datasets/ducanh2403/prompt-cache/prompt_clip_tokens_cache.pt')
VISUAL_CACHE=Path('/kaggle/input/datasets/ducanh2403/visual-cache')
DETECTIONS=Path('/kaggle/input/datasets/ducanh2403/objectdetectionecache/objectdetectioncache.json')
OLD_PREPARED=Path('/kaggle/working/concept_mass_'+ARM)
PREVIOUS_PREPARED=OLD_PREPARED if (OLD_PREPARED/'inputs.h5').is_file() and (OLD_PREPARED/'protocol.json').is_file() else None
OUTPUT=Path('/kaggle/working/concept_mass_'+ARM+'_v2')
REPORTED_REMAINING_GPU_HOURS=24


In [ ]:
SOURCE_COMMIT='33dbef35880c8482b3862b2c195b91729cafbe8b'
SOURCE_ASSETS={'experiments/concept_mass_grounding/concept_data.py': '733726ef9b03ea682f15a76b81017a22f6adb130d2cf130d0fa1d393fec2cf33', 'experiments/concept_mass_grounding/concept_model.py': 'c5c45d6ba506cd9fec99b406cf19793643a189bb23196bef1c96d6adb84714d0', 'experiments/concept_mass_grounding/prepare_concept.py': '532bb4d6d32df95387c696d71a86b10af92d06606070f2cc8e1b2e8405d2863a', 'experiments/concept_mass_grounding/run_concept.py': '1ab5a229902542dfef50c175cfcf5387f4e4489f1a6267d4ee31913518c86892', 'experiments/concept_mass_grounding/diagnostic_concept.py': '45286ed3e07006f331a6091ed67768b5f19fb31adfa4bf90f90099f12c9bb894', 'experiments/concept_mass_grounding/evaluate_concept.py': '04442924b17c543bfbbb6535de6cdbe64a78113e727662ce97e00dca706b0798', 'experiments/concept_mass_grounding/test_concept_data.py': '1dac8a7e1a7443b1fdd631f654606c9cfd40106be63f51dd02369f551812a8a3', 'experiments/concept_mass_grounding/recover_stage.py': 'd3d16ed410f7dc4ce0630a2dc74700642daed495134e010c37e8dc15c41699b4', 'experiments/concept_mass_grounding/test_recover_stage.py': 'd7892615af99c51dba31f57beab12ffc704f65b874198d706198e74b2688df8e', 'experiments/concept_mass_grounding/TRAIN_PRESENCE.json.gz': 'dd888949a1a8cc363efd760eb46012a7a6f736ac6b167d98be5303393b614117', 'experiments/concept_mass_grounding/DEV_DIAGNOSTIC_GT.json.gz': '4273bb0572e1d7358b3fb885d1bbb8505b29af5db9356adc3670ff33b21b3f01', 'experiments/shared_pair_readout/readout_data.py': '3bdc5e38c942cfeb13f8a83e409e367d6cba1b2fb5da6149c2d400b0c51b889c', 'experiments/shared_pair_readout/metric_preflight.py': '6e29f54e57b0b2e7f6f62de57017061071444d254b09305411716713b86f83f4', 'experiments/relation_frame_pretrain/frame_data.py': '87c58a50142f68b1721323f7c53fa08e256bb5bc8aee3d7625262161ae1692de', 'experiments/semantic_grounding_method/data/gain_metadata.json.gz': '62f43f771aec9feba31a2fbbb6fe7db2878f259979faac378b74a754822c7b67', 'experiments/coco_grounding_feasibility/MAPPING_POLICY_V11.json': 'fd13046512449449375f94ecda44237a33d5a9223813d036fc6d76f4294abfab', 'captioning/__init__.py': 'ad259f50a32265d298d027a888f801d378c4669313b8dab06e0653978ce30633', 'captioning/models.py': '18026877836052c83e75c0d896adcc89822bb35616173c7e8b14b17c9a22e9d8', 'captioning/config.py': '4766d1bf82bf82b5199616c7d90ca74242f011e26f08db21b154b1c7d58e72bc', 'captioning/grounded_relations.py': '2c6b15048dce17a9bfa596f02af78866f0b7978f65d172414a3648c9aa49c58f', 'captioning/decoder_relation_memory.py': 'bc5086915bf8f35cf7161311785cf612b25695c5f0dbe03a760d8aca724dfc82', 'captioning/scst.py': '3a9a1f32fb27dae0447066e1388a31461f3f0781d52db5861880b037a572a78e', 'captioning/scst_runtime.py': '3452ccfbd43876dc6381b78d1229d1d6b05fe723e3863a04f31c4a784497f669', 'captioning/tokenizer.py': 'df2ace0025aa7d4e12c289ab7bca68046b48d4b6c2b5c12606a19ff5a50a638e', 'captioning/object_reward.py': '23282b1faca0b8e177ad8ce300f0b78ad4a5ab7b100e4b359b044025c567800c', 'captioning/object_consistency.py': '5fbd75b0ff4a35f0475d460a294894009d6007f468e248a0e4b528aaa8fabf73', 'captioning/visual_cache.py': 'adbcc9678fff499fc767cdbf04a4966186a40729f51c5bfb2525e79f9b6b5fe1', 'captioning/semantic_prompts.py': 'a06f12dea63604f9db9d19b6922815db27a35b648a17059407ffaf26a2a0d511'}
import hashlib,json,os,signal,subprocess,sys,tempfile,threading,time,traceback,zipfile
os.environ.update(CUBLAS_WORKSPACE_CONFIG=':4096:8',OMP_NUM_THREADS='2',OPENBLAS_NUM_THREADS='2',
    PYTHONUNBUFFERED='1',PYTHONHASHSEED='42',TOKENIZERS_PARALLELISM='false',HDF5_USE_FILE_LOCKING='FALSE')
REPO=Path(tempfile.mkdtemp(prefix='concept-mass-'+ARM+'-'))/'repo'
LOG=OUTPUT.with_name(OUTPUT.name+'_notebook.log');REVIEW=OUTPUT.with_name(OUTPUT.name+'_review.zip')
for p in (OUTPUT,LOG,REVIEW):
    if p.exists():raise FileExistsError('Giữ output/log cũ; dùng phiên mới: '+str(p))
for p in (CHECKPOINT,COCO_JSON,LEGACY_PROMPT,DETECTIONS):
    if not p.is_file():raise FileNotFoundError('Add Input chứa '+str(p))
if not VISUAL_CACHE.is_dir():raise FileNotFoundError(VISUAL_CACHE)
def run(args,cwd=None,timeout=900):
    args=list(map(str,args));print('Running:',' '.join(args),flush=True)
    with LOG.open('a',encoding='utf8') as log:
        log.write('Running: '+' '.join(args)+'\n');log.flush()
        process=subprocess.Popen(args,cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,errors='replace',bufsize=1,start_new_session=True)
        def drain():
            for line in process.stdout:print(line,end='',flush=True);log.write(line);log.flush()
        reader=threading.Thread(target=drain,daemon=True);reader.start();start=time.monotonic()
        try:
            while process.poll() is None:
                if time.monotonic()-start>timeout:raise TimeoutError('Phase cap; incomplete')
                reader.join(timeout=20)
                if process.poll() is None:print('Worker active;',round(time.monotonic()-start),'s elapsed; see stage/step logs.',flush=True)
            reader.join()
            if process.returncode:raise subprocess.CalledProcessError(process.returncode,args)
        finally:
            if process.poll() is None:
                os.killpg(process.pid,signal.SIGTERM)
                try:process.wait(timeout=10)
                except subprocess.TimeoutExpired:os.killpg(process.pid,signal.SIGKILL);process.wait()
            reader.join(timeout=5)
print('Bootstrap started; input paths found.',flush=True)


In [ ]:
try:
    run(['git','init',REPO]);run(['git','remote','add','origin','https://github.com/Supzxjee/Image_Captioning.git'],cwd=REPO)
    run(['git','fetch','--depth','1','origin',SOURCE_COMMIT],cwd=REPO);run(['git','checkout','--detach','FETCH_HEAD'],cwd=REPO)
    if subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()!=SOURCE_COMMIT:raise ValueError('Wrong source commit')
    for name,h in SOURCE_ASSETS.items():
        if hashlib.sha256((REPO/name).read_bytes()).hexdigest()!=h:raise ValueError('Changed source '+name)
    os.environ['PYTHONPATH']=str(REPO);HERE=REPO/'experiments/concept_mass_grounding'
    run([sys.executable,'-m','pip','install','-q','transformers==4.57.1','pycocoevalcap==1.2','h5py','numpy','Pillow','nltk==3.9.2','future'])
    run([sys.executable,'-m','pip','install','-q','--no-deps','git+https://github.com/clips/pattern.git@af754685cca3713db0abc4f020f2e94467c19d85'])
    run([sys.executable,'-u','-c',"import nltk;from pattern.en import singularize;assert singularize('umbrellas')=='umbrella';assert nltk.download('punkt');assert nltk.download('punkt_tab')"])
    run(['java','-version']);CHAIR=REPO.parent/'chair'
    run(['git','init',CHAIR]);run(['git','remote','add','origin','https://github.com/Maxlinn/CHAIR-metric-standalone.git'],cwd=CHAIR)
    run(['git','fetch','--depth','1','origin','4087a26211aa2339b9a76307cb8f0321ef691d0a'],cwd=CHAIR);run(['git','checkout','--detach','FETCH_HEAD'],cwd=CHAIR)
    if hashlib.sha256((CHAIR/'chair.pkl').read_bytes()).hexdigest()!='73ce11225febb97df38c5bfad60b3ff6cc25852e9df9a1330673155b4be851c3':raise ValueError('Changed CHAIR cache')
    sys.path.insert(0,str(REPO));sys.path.insert(0,str(HERE))
    import importlib.util
    policy_spec=importlib.util.spec_from_file_location('concept_mass_policy_'+SOURCE_COMMIT,HERE/'concept_data.py')
    policy_module=importlib.util.module_from_spec(policy_spec);policy_spec.loader.exec_module(policy_module)
    POLICY=policy_module.POLICY
    CONFIG=REPO.parent/'config.json'
    cfg=dict(arm=ARM,checkpoint=str(CHECKPOINT),dataset_json=str(COCO_JSON),legacy_prompt=str(LEGACY_PROMPT),
        visual_cache=str(VISUAL_CACHE),detections=str(DETECTIONS),metadata=str(REPO/'experiments/semantic_grounding_method/data/gain_metadata.json.gz'),
        chair=str(CHAIR),output=str(OUTPUT),policy=POLICY,source_commit=SOURCE_COMMIT,source_assets=SOURCE_ASSETS,
        previous_prepared=str(PREVIOUS_PREPARED) if PREVIOUS_PREPARED is not None else None,
        reported_remaining_gpu_hours=REPORTED_REMAINING_GPU_HOURS,quota_note='User report24h; account distribution/billing not inferred; enabledGPU CPU phases extra')
    CONFIG.write_text(json.dumps(cfg,indent=2),encoding='utf8')

except BaseException:
    with zipfile.ZipFile(REVIEW,'w',zipfile.ZIP_DEFLATED) as z:
        if LOG.is_file():z.write(LOG,'notebook.log')
        z.writestr('execution_status.json',json.dumps(dict(success=False,arm=ARM,stage='bootstrap',source_commit=SOURCE_COMMIT)))
        z.writestr('bootstrap_error.txt',traceback.format_exc())
    print('BOOTSTRAP FAILED; SEND:',REVIEW,flush=True)
    raise


In [ ]:
success=False
try:
    run([sys.executable,'-u',HERE/'test_concept_data.py'],cwd=REPO)
    run([sys.executable,'-u',HERE/'test_recover_stage.py'],cwd=REPO)
    run([sys.executable,'-u',REPO/'experiments/shared_pair_readout/metric_preflight.py'],cwd=REPO)
    run([sys.executable,'-u',HERE/'run_concept.py','prepare','--config',CONFIG],cwd=REPO,timeout=3720)
    run([sys.executable,'-u','-m','torch.distributed.run','--standalone','--nnodes=1','--nproc_per_node=2','--max_restarts=0',
        HERE/'run_concept.py','worker','--config',CONFIG],cwd=REPO,timeout=3600)
    run([sys.executable,'-u',HERE/'run_concept.py','evaluate','--config',CONFIG],cwd=REPO,timeout=900)
    result=json.loads((OUTPUT/'summary.json').read_text())
    if result['status']!='ENDPOINT_500_EVALUATED_WAIT_MATCHED_C_G':raise RuntimeError('Incomplete evaluation')
    success=True
finally:
    with zipfile.ZipFile(REVIEW,'w',zipfile.ZIP_DEFLATED) as z:
        if LOG.is_file():z.write(LOG,'notebook.log')
        z.write(CONFIG,'notebook_config.json')
        for name in SOURCE_ASSETS:
            if name.endswith('.py'):z.write(REPO/name,'source/'+name)
        if OUTPUT.is_dir():
            for p in OUTPUT.iterdir():
                if p.suffix in ('.json','.jsonl','.log'):z.write(p,p.name)
        z.writestr('execution_status.json',json.dumps(dict(success=success,arm=ARM,source_commit=SOURCE_COMMIT,assistant_launched_job=False),indent=2))
    print('SEND SMALL REVIEW ZIP:',REVIEW,flush=True)
    if (OUTPUT/'checkpoint.bin').is_file():print('KEEP CHECKPOINT:',OUTPUT/'checkpoint.bin',flush=True)
    if not success:print('INCOMPLETE; send ZIP/log, do not accept as a method result.',flush=True)
if success:print('DONE',ARM,'; send BOTH C/G reviewZIPs for matched comparison, not automatic promotion.',flush=True)


Giữ `checkpoint.bin`. Gửi `concept_mass_C_v2_review.zip` và `concept_mass_G_v2_review.zip`.
Hai notebook chưa được assistant chạy. Actual gradients/DDP/numeric parity chỉ được kiểm ở preflight user-run.
Endpoint chỉ là pilot; G cần có tín hiệu tốt hơn A cả CIDEr lẫn CHAIR, có ích so C và giữ object recall.
Không tự đổi lambda, chọn checkpoint, chạy full/seed2 hoặc upload Dataset.
